# 04 - ML Model Training & Hyperparameter Selection
Demonstrates model training, 5-fold Stratified K-Fold cross-validation, hyperparameter tuning, class imbalance handling, and candidate comparison on `data/processed/financial_fraud_processed.csv`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config.settings import settings
from src.features.feature_pipeline import create_preprocessing_pipeline
from src.models.train import train_all_models, RANDOM_STATE
from src.models.model_registry import ModelRegistry


## 1. Load Processed Dataset & Exclude Identifiers
Load feature-ready data and split target `Fraudulent` from predictive feature set.


In [ ]:
data_path = ROOT_DIR / 'data' / 'processed' / 'financial_fraud_processed.csv'
df = pd.read_csv(data_path)
print(f'Processed Dataset Shape: {df.shape}')

target_col = 'Fraudulent'
drop_cols = ['Transaction_ID', 'Customer_ID', 'Transaction_Date', 'Suspicious_Keyword', 'Is_International', target_col]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df[target_col]
print(f'Feature Set Count: {len(feature_cols)} features')


## 2. Stratified Train / Test Split (80/20)
Split dataset into 80% training set and 20% test set while preserving minority fraud ratio.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Training Set: {len(X_train)} rows ({y_train.sum()} fraud)')
print(f'Testing Set: {len(X_test)} rows ({y_test.sum()} fraud)')


## 3. Fit ColumnTransformer Preprocessing Pipeline
Fit numerical scaler and categorical encoder ONLY on training fold.


In [ ]:
preprocessor = create_preprocessing_pipeline()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print(f'Transformed Matrix Shape: {X_train_proc.shape}')


## 4. Execute 5-Fold Stratified CV & Model Tuning
Train and tune Logistic Regression, Random Forest, and XGBoost models.


In [ ]:
models_dict = train_all_models(X_train_proc, y_train)
for name, item in models_dict.items():
    print(f'{name} Best CV PR-AUC: {item["cv_pr_auc"]:.4f}')
